# A3-M2-2 · RAG평가와 리랭킹
교안의 같은 번호 단원과 함께 봅니다. 노트북은 01 → 02 → 03 순서로 실행합니다.

## 06-1. 준비 — 저장한 검색기 열기

In [1]:
import json, time
import numpy as np
import pandas as pd
import faiss
from emb import embed, EMB_MODEL, RERANK_MODEL

index = faiss.read_index("faq_store/index.faiss")
meta = pd.read_csv("faq_store/meta.csv").set_index("faq_id")
man = json.load(open("faq_store/manifest.json", encoding="utf-8"))
assert man["model"] == EMB_MODEL, "다른 모델로 만든 인덱스입니다"
tests = pd.read_csv("shop_faq_tests.csv")

def ids_of(categories):
    return meta.index[meta["category"].isin(categories)].to_numpy(dtype="int64")

def search(question, k=3, categories=None):
    params = None
    if categories:
        params = faiss.SearchParameters(sel=faiss.IDSelectorBatch(ids_of(categories)))
    D, I = index.search(embed([question], kind="query"), k, params=params)
    hits = [(int(i), float(d)) for d, i in zip(D[0], I[0]) if i != -1]
    res = meta.loc[[i for i, _ in hits], ["category", "question", "answer"]].copy()
    res.insert(0, "score", [round(s, 3) for _, s in hits])
    return res

print("FAQ", index.ntotal, "건 | 평가 질문", len(tests), "개 | 모델", man["model"])

FAQ 20 건 | 평가 질문 12 개 | 모델 intfloat/multilingual-e5-small


## 06-2. 평가 질문 12개로 검색 평가 — Hit@1 · Hit@3 · MRR

In [2]:
rows = []
for q, gold in tests.itertuples(index=False):
    res = search(q, k=5)
    ranked = res.index.tolist()
    rank = ranked.index(gold) + 1 if gold in ranked else np.nan
    rows.append({"question": q, "gold": gold, "top1": ranked[0], "rank": rank, "top1_score": res["score"].iloc[0]})
ev = pd.DataFrame(rows)
print(ev.assign(rank=ev["rank"].astype("Int64")).to_string(index=False))

hit1 = (ev["rank"] == 1).mean()
hit3 = (ev["rank"] <= 3).mean()
mrr = (1 / ev["rank"]).fillna(0).mean()
print(f"\nHit@1 {hit1:.3f}  Hit@3 {hit3:.3f}  MRR {mrr:.3f}")

c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1349.27it/s]


            question  gold  top1  rank  top1_score
           택배 언제 와요?   101   101     1       0.846
     배송비 무료 기준이 뭐예요?   102   102     1       0.904
내 택배 지금 어디쯤인지 보고 싶어요   103   103     1       0.874
 소파 같은 큰 물건은 어떻게 와요?   104   104     1       0.821
     미국으로 보내줄 수 있나요?   105   105     1       0.866
    카드 할부 몇 개월까지 돼요?   202   202     1       0.901
  받은 지 5일 됐는데 반품 돼요?   301   301     1       0.892
         물건이 깨져서 왔어요   303   303     1       0.860
          환불 며칠 걸려요?   401   401     1       0.885
       주문 취소 어떻게 해요?   501   501     1       0.884
            비번 까먹었어요   601   303     3       0.823
       포인트 유효기간 있나요?   701   701     1       0.846

Hit@1 0.917  Hit@3 1.000  MRR 0.944


## 06-3. 틀린 질문 들여다보기

In [3]:
miss = ev[ev["rank"] != 1]
print(f"1위가 정답이 아닌 질문 {len(miss)}개")
for r in miss.itertuples(index=False):
    print(f"\nQ: {r.question}")
    print(f"  정답 {r.gold} [{meta.loc[r.gold, 'category']}] {meta.loc[r.gold, 'question']}  (정답 순위 {r.rank})")
    print(f"  1위  {r.top1} [{meta.loc[r.top1, 'category']}] {meta.loc[r.top1, 'question']}")

1위가 정답이 아닌 질문 1개

Q: 비번 까먹었어요
  정답 601 [회원] 비밀번호를 잊어버렸어요.  (정답 순위 3)
  1위  303 [교환반품] 상품이 파손되어 왔어요.


## 06-4. 답하지 않을 기준 정하기 — 점수 분포 보기

In [4]:
off = ["오늘 날씨 어때요?", "주식 추천해 주세요", "점심 메뉴 골라줘", "파이썬 설치 방법", "축구 경기 결과 알려줘", "영화 추천해 줘"]
ok = sorted(ev["top1_score"].round(3).tolist())
bad = sorted(round(float(search(q, 1)["score"].iloc[0]), 3) for q in off)
print("쇼핑 질문 1위 점수:", ok)
print("상관없는 질문 점수:", bad)
THRESHOLD = round((min(ok) + max(bad)) / 2, 3)
print("기준값 후보:", THRESHOLD, "| 겹침" if min(ok) <= max(bad) else "| 두 분포가 나뉨")

쇼핑 질문 1위 점수: [0.821, 0.823, 0.846, 0.846, 0.86, 0.866, 0.874, 0.884, 0.885, 0.892, 0.901, 0.904]
상관없는 질문 점수: [0.795, 0.81, 0.816, 0.827, 0.831, 0.833]
기준값 후보: 0.827 | 겹침


## 06-5. LLM에 넘길 프롬프트 만들기 (RAG의 R까지)

In [5]:
def build_prompt(question, k=3, categories=None):
    res = search(question, k, categories)
    if res["score"].iloc[0] < THRESHOLD:
        return None
    ctx = "\n".join(f"[{i}] ({r.category}) Q: {r.question} A: {r.answer}" for i, r in res.iterrows())
    return f"다음 FAQ만 근거로 답하세요. 답 끝에 근거 번호를 적으세요.\n\n{ctx}\n\n질문: {question}"

print(build_prompt("물건이 깨져서 왔어요"))
print("\n오늘 날씨 어때요? →", build_prompt("오늘 날씨 어때요?") or "상담원 연결(근거 부족)")

다음 FAQ만 근거로 답하세요. 답 끝에 근거 번호를 적으세요.

[303] (교환반품) Q: 상품이 파손되어 왔어요. A: 파손·불량 상품은 사진과 함께 접수하시면 배송비 없이 교환 또는 반품해 드립니다.
[301] (교환반품) Q: 교환이나 반품은 언제까지 되나요? A: 상품을 받은 날부터 7일 이내에 마이페이지에서 신청할 수 있습니다.
[602] (회원) Q: 회원 탈퇴는 어떻게 하나요? A: 마이페이지 > 회원 정보 > 회원 탈퇴에서 신청할 수 있으며 적립금은 소멸됩니다.

질문: 물건이 깨져서 왔어요

오늘 날씨 어때요? → 다음 FAQ만 근거로 답하세요. 답 끝에 근거 번호를 적으세요.

[301] (교환반품) Q: 교환이나 반품은 언제까지 되나요? A: 상품을 받은 날부터 7일 이내에 마이페이지에서 신청할 수 있습니다.
[105] (배송) Q: 해외로도 배송되나요? A: 현재는 국내 배송만 가능하며 해외 배송은 지원하지 않습니다.
[101] (배송) Q: 배송은 언제 오나요? A: 주문 후 평일 기준 2~3일 이내에 도착합니다. 제주·도서산간은 3~5일 걸립니다.

질문: 오늘 날씨 어때요?


## 07-1. 리랭커 써 보기 — 질문과 문서를 한 쌍으로 넣는 모델

In [6]:
from transformers import pipeline
rr = pipeline("text-classification", model=RERANK_MODEL, device=-1)

q = "물건이 깨져서 왔어요"
cand = search(q, k=5)
pairs = [{"text": q, "text_pair": f"{r.question} {r.answer}"} for r in cand.itertuples()]
scores = rr(pairs, function_to_apply="none")
print("모델:", RERANK_MODEL)
print("출력 예:", scores[0])
cand["rerank"] = [round(s["score"], 3) for s in scores]
print("\n[FAISS 순서]")
print(cand[["score", "rerank", "category", "question"]].to_string())
print("\n[리랭킹 순서]")
print(cand.sort_values("rerank", ascending=False)[["score", "rerank", "category", "question"]].to_string())

c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--cross-encoder--mmarco-mMiniLMv2-L12-H384-v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-package

모델: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
출력 예: {'label': 'LABEL_0', 'score': 0.7158318161964417}

[FAISS 순서]
        score  rerank category            question
faq_id                                            
303     0.860   0.716     교환반품       상품이 파손되어 왔어요.
301     0.833  -7.350     교환반품  교환이나 반품은 언제까지 되나요?
602     0.827  -9.832       회원     회원 탈퇴는 어떻게 하나요?
501     0.825  -6.085       주문       주문을 취소하고 싶어요.
101     0.823  -9.121       배송         배송은 언제 오나요?

[리랭킹 순서]
        score  rerank category            question
faq_id                                            
303     0.860   0.716     교환반품       상품이 파손되어 왔어요.
501     0.825  -6.085       주문       주문을 취소하고 싶어요.
301     0.833  -7.350     교환반품  교환이나 반품은 언제까지 되나요?
101     0.823  -9.121       배송         배송은 언제 오나요?
602     0.827  -9.832       회원     회원 탈퇴는 어떻게 하나요?


## 07-2. 검색 + 리랭킹 함수로 다시 평가

In [7]:


def search_rerank(question, k=3, pool=8, categories=None):
    cand = search(question, k=pool, categories=categories)              # 1단계: FAISS로 후보
    pairs = [{"text": question, "text_pair": f"{r.question} {r.answer}"} for r in cand.itertuples()]
    cand["rerank"] = [round(s["score"], 3) for s in rr(pairs, function_to_apply="none")]
    return cand.sort_values("rerank", ascending=False).head(k)          # 2단계: 다시 줄 세우기

def evaluate(fn, **kw):
    rows = []
    for q, gold in tests.itertuples(index=False):
        ranked = fn(q, **kw).index.tolist()
        rows.append({"question": q, "gold": gold, "top1": ranked[0],
                     "rank": ranked.index(gold) + 1 if gold in ranked else np.nan})
    return pd.DataFrame(rows)

def summary(ev):
    return {"Hit@1": round((ev["rank"] == 1).mean(), 3), "Hit@3": round((ev["rank"] <= 3).mean(), 3),
            "MRR": round((1 / ev["rank"]).fillna(0).mean(), 3)}

ev_base = evaluate(search, k=5)
ev_rr = evaluate(search_rerank, k=5, pool=8)
print(pd.DataFrame({"검색만": summary(ev_base), "검색+리랭킹": summary(ev_rr)}).to_string())

         검색만  검색+리랭킹
Hit@1  0.917   0.833
Hit@3  1.000   1.000
MRR    0.944   0.917


## 07-3. 어떤 질문이 바뀌었나 · 시간은 얼마나 드나

In [8]:
both = ev_base[["question", "gold", "rank"]].merge(ev_rr[["question", "rank"]], on="question", suffixes=("_검색", "_리랭킹"))
both["변화"] = np.where(both["rank_리랭킹"] < both["rank_검색"], "↑", np.where(both["rank_리랭킹"] > both["rank_검색"], "↓", ""))
print(both.astype({"rank_검색": "Int64", "rank_리랭킹": "Int64"}).to_string(index=False))

for name, fn in [("검색만", lambda q: search(q, 5)), ("검색+리랭킹", lambda q: search_rerank(q, 5, 8))]:
    t = time.perf_counter()
    for q in tests["question"]:
        fn(q)
    print(f"{name:<8} 질문당 {(time.perf_counter() - t) / len(tests) * 1000:.0f} ms")

            question  gold  rank_검색  rank_리랭킹 변화
           택배 언제 와요?   101        1         1   
     배송비 무료 기준이 뭐예요?   102        1         1   
내 택배 지금 어디쯤인지 보고 싶어요   103        1         2  ↓
 소파 같은 큰 물건은 어떻게 와요?   104        1         1   
     미국으로 보내줄 수 있나요?   105        1         1   
    카드 할부 몇 개월까지 돼요?   202        1         1   
  받은 지 5일 됐는데 반품 돼요?   301        1         1   
         물건이 깨져서 왔어요   303        1         1   
          환불 며칠 걸려요?   401        1         1   
       주문 취소 어떻게 해요?   501        1         1   
            비번 까먹었어요   601        3         2  ↑
       포인트 유효기간 있나요?   701        1         1   
검색만      질문당 47 ms
검색+리랭킹   질문당 521 ms
